In [16]:
# =============================================================================
# Data Processing
# =============================================================================
import pandas as pd
import numpy as np


import re

# =============================================================================
# Embedding Models
# =============================================================================
from sentence_transformers import SentenceTransformer

In [17]:
# =============================================================================
# Load semantic_dataset documents
# =============================================================================
semantic_dataset = pd.read_csv("semantic_chunks.csv")
print(semantic_dataset.shape)
semantic_dataset.head()

(113871, 8)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C15Q2112_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,Question: - what are the dimensions of this it...,Category: Tools and Home Improvement Question ...
1,C9Q4595_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,Question: how much booze can it hold? Answer: ...,Category: Home and Kitchen Question Type: open...
2,C4Q7999_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,Question: will this case fit nokia lumia 520 A...,Category: Cell Phones and Accessories Question...
3,C8Q8916_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,"Question: when folded in the sitting position,...",Category: Health and Personal Care Question Ty...
4,C14Q905_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,Question: how long should i leave this on to g...,Category: Sports and Outdoors Question Type: o...


In [18]:
semantic_dataset.columns

Index(['chunk_id', 'QuestionID', 'Category', 'QuestionType', 'QuestionTime',
       'chunk_index', 'chunk_text', 'search_text'],
      dtype='str')

In [19]:
# =============================================================================
# Load lexical_dataset documents
# =============================================================================
lexical_dataset = pd.read_csv("lexical_chunks.csv")
print(lexical_dataset.shape)
lexical_dataset.head()

(105096, 8)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,0_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,dimensions item,Tools and Home Improvement open-ended dimensio...
1,1_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,much booze hold poured booze measuring cup sun...,Home and Kitchen open-ended much booze hold po...
2,2_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,case fit nokia lumia 520 yes fits great great ...,Cell Phones and Accessories open-ended case fi...
3,3_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,folded sitting position high ground seat 30 in...,Health and Personal Care open-ended folded sit...
4,4_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,long leave get max sweat benefit keep thinking...,Sports and Outdoors open-ended long leave get ...


In [20]:
lexical_dataset.columns

Index(['chunk_id', 'QuestionID', 'Category', 'QuestionType', 'QuestionTime',
       'chunk_index', 'chunk_text', 'search_text'],
      dtype='str')

In [21]:
# =============================================================================
# Load SentenceTransformer Model
# =============================================================================
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [22]:
# =============================================================================
# Prepare Semantic Documents
# =============================================================================
documents = semantic_dataset["search_text"].tolist()
print("Total Chunks:", len(documents))

Total Chunks: 113871


In [ ]:
# =============================================================================
# Generate Sentence Embeddings
# =============================================================================
embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True)

print(embeddings.shape)

Batches:   0%|          | 0/3559 [00:00<?, ?it/s]

In [ ]:
# =============================================================================
# Store Embeddings
# =============================================================================
semantic_dataset["embedding"] = embeddings.tolist()
semantic_dataset.head()

In [ ]:
# =============================================================================
# BM25 Tokenizer
# =============================================================================
def tokenize_text(text):

    """
    Tokenize text for BM25 indexing.
    """

    return re.findall(
        r"\b[a-zA-Z0-9]+\b",
        str(text).lower())

In [ ]:
# =============================================================================
# Tokenize Lexical Documents
# =============================================================================
lexical_dataset["lexical_tokens"] = (lexical_dataset["search_text"].fillna("").apply(tokenize_text))
lexical_dataset.head()

In [ ]:
print(lexical_dataset.loc[0, "lexical_tokens"])

In [ ]:
# =============================================================================
# Save embedding matrix
# =============================================================================
np.save("semantic_embeddings.npy", embeddings)
print("Embeddings saved successfully.")

In [ ]:
# =============================================================================
# Save Metadata of semantic dataset
# =============================================================================
semantic_dataset.to_csv("semantic_chunks.csv",index=False)
print("Semantic dataset saved successfully.")

In [ ]:
# =============================================================================
# Save Lexical Dataset for BM25
# =============================================================================
lexical_dataset.to_csv("lexical_tokens.csv", index=False)
print("Lexical dataset saved successfully.")